# 25 — FIM Scaling Law: λ_c(N)

Notebook 24 showed FIM (strange loop) solves N=16 synchronisation.
The critical question: **how does the required FIM strength scale with system size?**

If λ_c(N) ~ N^α, the exponent α tells us:
- α < 1: self-observation becomes relatively CHEAPER at large N (favourable)
- α = 1: linear scaling (neutral)
- α > 1: super-linear — large systems need disproportionately more self-observation

This is a universal prediction of SCPN that no other framework makes.

## Method
For each N ∈ {4,6,8,10,12,14,16,18,20,24}:
1. Fix K_scale at the coupling-only sync threshold (R=0.5 without FIM)
2. Binary search for λ_c such that R ≥ 0.8
3. Also measure at fixed K_scale values for comparison
4. Fit power law λ_c(N) = a * N^α

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
from scipy.optimize import curve_fit

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27


def get_omega(N):
    """Get natural frequencies for N oscillators. Extend beyond 16 with same distribution."""
    if N <= 16:
        return OMEGA_N_16[:N].copy()
    rng = np.random.default_rng(12345)
    extra = rng.uniform(OMEGA_N_16.min(), OMEGA_N_16.max(), N - 16)
    return np.concatenate([OMEGA_N_16, extra])


def fim_gradient(phases, i):
    """FIM strange loop gradient for oscillator i."""
    N = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases[i] + np.pi) % (2 * np.pi) - np.pi
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    return (1.0 / N) * np.sin(phase_diff) * min(sensitivity, 50.0)


def simulate_R(N, K_scale, fim_lambda, dt=0.02, T=150.0, noise=0.05, seed=42):
    """Simulate Kuramoto + FIM, return time-averaged R over last 25%."""
    K = build_knm_paper27(L=min(N, 16))
    if N > 16:
        # Extend K_nm with exponential decay for extra oscillators
        K_full = np.zeros((N, N))
        K_full[:16, :16] = K
        for i in range(16, N):
            for j in range(N):
                if i != j:
                    dist = abs(i - j)
                    K_full[i, j] = K_full[j, i] = 0.45 * np.exp(-0.3 * dist)
        K = K_full
    K = K * K_scale
    omega = get_omega(N)
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)

    R_tail = []
    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if fim_lambda > 0:
            for i in range(N):
                dphi[i] += fim_lambda * fim_gradient(theta, i)
        theta += dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)
        theta %= 2 * np.pi
        if s >= n_steps * 3 // 4:
            R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))

    return float(np.mean(R_tail))


def simulate_R_multi(N, K_scale, fim_lambda, n_seeds=5, **kwargs):
    """Average R over multiple seeds."""
    Rs = [simulate_R(N, K_scale, fim_lambda, seed=s, **kwargs) for s in range(n_seeds)]
    return float(np.mean(Rs)), float(np.std(Rs))


print("Functions defined.")

## 1. Find Coupling-Only Threshold K_c(N) Without FIM

In [ ]:
N_values = [4, 6, 8, 10, 12, 14, 16, 18, 20]
R_TARGET = 0.8

# First: find K_c(N) without FIM (binary search)
K_c_no_fim = {}
for N in N_values:
    lo, hi = 0.5, 30.0
    for _ in range(15):  # 15 iterations of binary search
        mid = (lo + hi) / 2
        R, _ = simulate_R_multi(N, mid, 0.0, n_seeds=3)
        if R >= R_TARGET:
            hi = mid
        else:
            lo = mid
    K_c_no_fim[N] = round((lo + hi) / 2, 2)
    print(f"N={N:2d}: K_c(no FIM) = {K_c_no_fim[N]:.2f}")

print("\nK_c grows with N — more coupling needed for larger systems.")

## 2. Find λ_c(N) at Fixed K_scale

For each N, fix K_scale at 70% of K_c(N) (sub-threshold coupling),
then binary search for λ_c that achieves R ≥ 0.8.
This measures how much FIM is needed to compensate for insufficient coupling.

In [ ]:
lambda_c_results = {}
K_FRACTION = 0.7  # 70% of coupling-only threshold

for N in N_values:
    K_scale = K_c_no_fim[N] * K_FRACTION

    # Binary search for lambda_c
    lo, hi = 0.0, 50.0

    # First check if FIM can solve it at all
    R_max, _ = simulate_R_multi(N, K_scale, hi, n_seeds=3)
    if R_max < R_TARGET:
        print(f"N={N:2d}: FIM cannot reach R>={R_TARGET} even at λ={hi} (max R={R_max:.3f})")
        lambda_c_results[N] = {
            "K_scale": round(K_scale, 2),
            "lambda_c": None,
            "R_max": round(R_max, 3),
        }
        continue

    for _ in range(20):
        mid = (lo + hi) / 2
        R, _ = simulate_R_multi(N, K_scale, mid, n_seeds=3)
        if R >= R_TARGET:
            hi = mid
        else:
            lo = mid

    lam_c = (lo + hi) / 2
    R_check, R_std = simulate_R_multi(N, K_scale, lam_c, n_seeds=5)
    lambda_c_results[N] = {
        "K_scale": round(K_scale, 2),
        "lambda_c": round(lam_c, 3),
        "R_at_lambda_c": round(R_check, 4),
    }
    print(f"N={N:2d}: K_scale={K_scale:.2f}, λ_c={lam_c:.3f}, R={R_check:.4f}±{R_std:.4f}")

## 3. Power Law Fit: λ_c(N) = a · N^α

In [ ]:
# Extract valid data points
N_fit = []
lam_fit = []
for N in N_values:
    if lambda_c_results[N]["lambda_c"] is not None:
        N_fit.append(N)
        lam_fit.append(lambda_c_results[N]["lambda_c"])

N_fit = np.array(N_fit, dtype=float)
lam_fit = np.array(lam_fit)

if len(N_fit) >= 3:
    # Power law fit
    def power_law(x, a, alpha):
        return a * x**alpha

    popt, pcov = curve_fit(power_law, N_fit, lam_fit, p0=[0.1, 1.0])
    a, alpha = popt
    perr = np.sqrt(np.diag(pcov))

    # Goodness of fit
    lam_pred = power_law(N_fit, *popt)
    ss_res = np.sum((lam_fit - lam_pred) ** 2)
    ss_tot = np.sum((lam_fit - np.mean(lam_fit)) ** 2)
    r_squared = 1 - ss_res / ss_tot

    print(f"Power law fit: λ_c(N) = {a:.4f} · N^{alpha:.4f}")
    print(f"  a = {a:.4f} ± {perr[0]:.4f}")
    print(f"  α = {alpha:.4f} ± {perr[1]:.4f}")
    print(f"  R² = {r_squared:.6f}")
    print()

    if alpha < 1:
        print(f"α = {alpha:.2f} < 1: Self-observation becomes RELATIVELY CHEAPER at large N.")
        print("This is favourable — consciousness scales sub-linearly.")
    elif alpha > 1:
        print(f"α = {alpha:.2f} > 1: Self-observation cost grows SUPER-LINEARLY.")
        print("Large systems need disproportionately more strange loop feedback.")
    else:
        print("α ≈ 1: Linear scaling.")

    # Also try log fit
    def log_law(x, a, b):
        return a * np.log(x) + b

    try:
        popt_log, pcov_log = curve_fit(log_law, N_fit, lam_fit, p0=[1.0, 0.0])
        lam_pred_log = log_law(N_fit, *popt_log)
        ss_res_log = np.sum((lam_fit - lam_pred_log) ** 2)
        r2_log = 1 - ss_res_log / ss_tot
        print(f"\nLogarithmic fit: λ_c(N) = {popt_log[0]:.4f} · ln(N) + {popt_log[1]:.4f}")
        print(f"  R² = {r2_log:.6f}")
        print(f"  Better fit: {'logarithmic' if r2_log > r_squared else 'power law'}")
    except Exception:
        pass

    # Print comparison table
    print(f"\n{'N':>4} {'λ_c (measured)':>15} {'λ_c (power law)':>16} {'residual':>10}")
    for i, N in enumerate(N_fit):
        print(
            f"{int(N):4d} {lam_fit[i]:15.3f} {lam_pred[i]:16.3f} {lam_fit[i] - lam_pred[i]:10.3f}"
        )

    # Extrapolation
    print("\n=== Extrapolation ===")
    for N_ext in [32, 64, 128, 256]:
        lam_ext = power_law(N_ext, *popt)
        print(f"N={N_ext:3d}: λ_c ≈ {lam_ext:.2f}")
else:
    print(f"Only {len(N_fit)} data points — not enough for fit.")

## 4. Alternative: λ_c at Fixed K_scale Across All N

In [ ]:
# Fixed K_scale = 5.0 for all N — how much FIM do we need?
K_FIXED = 5.0
lambda_c_fixed = {}

print(f"Fixed K_scale = {K_FIXED}")
for N in N_values:
    # Binary search for lambda_c
    lo, hi = 0.0, 50.0
    R_max, _ = simulate_R_multi(N, K_FIXED, hi, n_seeds=3)
    if R_max < R_TARGET:
        print(f"N={N:2d}: Cannot reach R>={R_TARGET} (max R={R_max:.3f})")
        lambda_c_fixed[N] = None
        continue

    for _ in range(20):
        mid = (lo + hi) / 2
        R, _ = simulate_R_multi(N, K_FIXED, mid, n_seeds=3)
        if R >= R_TARGET:
            hi = mid
        else:
            lo = mid

    lam_c = (lo + hi) / 2
    lambda_c_fixed[N] = round(lam_c, 3)
    print(f"N={N:2d}: λ_c = {lam_c:.3f}")

# Fit this too
N_f2 = np.array([N for N in N_values if lambda_c_fixed[N] is not None], dtype=float)
l_f2 = np.array([lambda_c_fixed[N] for N in N_values if lambda_c_fixed[N] is not None])

if len(N_f2) >= 3:
    popt2, _ = curve_fit(power_law, N_f2, l_f2, p0=[0.1, 1.0])
    print(f"\nPower law (K_fixed={K_FIXED}): λ_c(N) = {popt2[0]:.4f} · N^{popt2[1]:.4f}")

## 5. K_c Reduction Factor from FIM

In [ ]:
# How much does FIM reduce the coupling threshold?
# For each N, compute K_c(λ=1) and compare with K_c(λ=0)
K_c_with_fim = {}
for N in N_values:
    lo, hi = 0.5, 30.0
    for _ in range(15):
        mid = (lo + hi) / 2
        R, _ = simulate_R_multi(N, mid, 1.0, n_seeds=3)
        if R >= R_TARGET:
            hi = mid
        else:
            lo = mid
    K_c_with_fim[N] = round((lo + hi) / 2, 2)

print(f"{'N':>4} {'K_c(λ=0)':>10} {'K_c(λ=1)':>10} {'Reduction':>10} {'Factor':>8}")
for N in N_values:
    k0 = K_c_no_fim[N]
    k1 = K_c_with_fim[N]
    reduction = k0 - k1
    factor = k0 / k1 if k1 > 0 else float("inf")
    print(f"{N:4d} {k0:10.2f} {k1:10.2f} {reduction:10.2f} {factor:8.2f}x")

# Is the reduction factor constant, growing, or shrinking with N?
factors = [K_c_no_fim[N] / K_c_with_fim[N] for N in N_values if K_c_with_fim[N] > 0]
print(f"\nMean reduction factor: {np.mean(factors):.2f}x")
print(f"Trend: {'GROWING' if factors[-1] > factors[0] else 'SHRINKING'} with N")

In [ ]:
# Save all results
output = {
    "experiment": "FIM scaling law lambda_c(N)",
    "R_target": R_TARGET,
    "N_values": N_values,
    "K_c_no_fim": K_c_no_fim,
    "K_c_with_fim_lambda1": K_c_with_fim,
    "lambda_c_at_70pct_K": lambda_c_results,
    "lambda_c_at_K_fixed_5": {str(k): v for k, v in lambda_c_fixed.items()},
}
if len(N_fit) >= 3:
    output["power_law_fit"] = {
        "a": round(float(a), 4),
        "alpha": round(float(alpha), 4),
        "R_squared": round(float(r_squared), 6),
        "description": f"lambda_c(N) = {a:.4f} * N^{alpha:.4f}",
    }

out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/fim_scaling_law_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"Results saved to {out_path}")